# NeuralAI-Air-135M-SFT v18

Expanded supervised fine-tune of the custom 135M base on 500+ ChatML-formatted instruction/response pairs.

**Goal**: improve coherence over v17 (37 pairs).


In [ ]:
# === Clone repo ===
!git clone -b master https://github.com/Subject-Emu-5259/NeuralAI.git
%cd NeuralAI
!pip install -q transformers datasets accelerate peft bitsandbytes

In [ ]:
from pathlib import Path
from datasets import Dataset
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_PATH = '/home/.z/workspaces/con_Be6MM5KUzfA88RWI/neuralair-135m/neuralair-135m/final.pt'  # upload to this path in Colab
MODEL_NAME = 'Subject-Emu-5259/NeuralAI-Air-135M-SFT'  # reuse tokenizer/config from HF
DATA_PATH  = 'data/train_sft_v18.jsonl'
OUT_DIR    = 'checkpoints/v18-sft'

# Load tokenizer and model architecture
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(DEVICE)

# Load base weights if uploaded
base_pt = Path(BASE_PATH)
if base_pt.exists():
    model.load_state_dict(torch.load(str(base_pt), map_location=DEVICE))
    print('Loaded base checkpoint:', base_pt)
else:
    print('WARNING: base checkpoint not found. Continuing from HF init weights.')

In [ ]:
# Dataset preprocessing: tokenize full prompt+response, mask prompt tokens in labels
MAX_LEN = 512
IGNORE  = -100

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def tokenize(examples):
    texts  = examples['text']
    inputs = tokenizer(texts, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')
    labels = inputs['input_ids'].clone()
    # Mask everything before the assistant response for loss computation
    for i, txt in enumerate(texts):
        assistant_marker = 'assistant'  # chat string already has <...>assistant\n
        # Find index where assistant generation begins in token space
        prompt_end_text = assistant_marker + "\n"
        prompt_tok = tokenizer(prompt_end_text, add_special_tokens=False)['input_ids']
        prompt_len = len(tokenizer(txt[:txt.rfind(prompt_end_text)], add_special_tokens=False)['input_ids']) + len(prompt_tok)
        prompt_len = min(prompt_len, MAX_LEN)
        labels[i, :prompt_len] = IGNORE
    inputs['labels'] = labels
    return inputs

rows = load_jsonl(DATA_PATH)
dataset = Dataset.from_list(rows)
tokenized = dataset.map(lambda x: tokenize(x), batched=True, remove_columns=dataset.column_names)
print('Samples:', len(tokenized))

In [ ]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    optim='adamw_torch',
    report_to='none',
    remove_unused_columns=False,
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized, tokenizer=tokenizer)
trainer.train()
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

In [ ]:
# Push to Hugging Face (use Secret HF_TOKEN or login via huggingface-cli)
from huggingface_hub import HfApi
api = HfApi()
api.create_repo('Subject-Emu-5259/NeuralAI-Air-135M-SFT-v18', exist_ok=True)
model.push_to_hub('Subject-Emu-5259/NeuralAI-Air-135M-SFT-v18')
tokenizer.push_to_hub('Subject-Emu-5259/NeuralAI-Air-135M-SFT-v18')

In [ ]:
# Quick sanity generation
inputs = tokenizer('Who are you?', return_tensors='pt').to(DEVICE)
out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))